# 01 — Revisione bottom-up

Il notebook revisiona le baseline top-down esistenti. Ricostruisce HLG′ dai soli LLG, calcola le cosine e raccoglie le decisioni documentali di Ministral, Granite e Gemma tramite un prompt globale per modello. Il quorum richiede due voti validi concordi su tre.

La bottom-up non esegue audit diretti degli LLG o diagnosi della fedeltà della ricostruzione. La qualità degli obiettivi generati resta affidata ai due critic originali. Le correzioni approvate riutilizzano generazione e reflection originali e vengono rivalutate in una nuova iterazione.

Ogni progetto ha una cartella sotto `output/bottom_up_iterations/`, con un JSON delle cosine e uno dei risultati per iterazione. La pulizia avviene soltanto all'avvio di una nuova esecuzione. La baseline top-down resta invariata.


In [1]:
from pathlib import Path
import json
import re
import sys
import time

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

NOTEBOOK_DIR = PROJECT_ROOT / "notebook"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from groundtruth import ALL_GROUNDTRUTHS
import src.llm_clients as llm_clients
GROUNDTRUTH_NAMES = {item["name"] for item in ALL_GROUNDTRUTHS}

OUTPUT_ROOT = PROJECT_ROOT / "output"
TOP_DOWN_DIR = OUTPUT_ROOT / "top_down"
TOP_DOWN_BOTTOM_UP_DIR = OUTPUT_ROOT / "top_down_bottom_up"
ITERATIONS_ROOT = OUTPUT_ROOT / "bottom_up_iterations"

for path in (
    TOP_DOWN_DIR,
    TOP_DOWN_BOTTOM_UP_DIR,
    ITERATIONS_ROOT,
):
    path.mkdir(parents=True, exist_ok=True)

# Run all registered datasets using their existing top-down baselines.
# Start execution manually from this notebook.
DATASETS_TO_RERUN = [item["name"] for item in ALL_GROUNDTRUTHS]

BASELINE_FILE_OVERRIDES = [
    TOP_DOWN_DIR
    / f"{name}_ZERO_SHOT_feedback_original_top_down_top_down.json"
    for name in DATASETS_TO_RERUN
]

# One global documentary review per model, after numerical similarity.

# Maximum number of feedback iterations. By default use the orchestrator constant.
GLOBAL_CYCLE_MAX_ITERATIONS_OVERRIDE = 25

# Timeout for local inference, including reasoning.
REQUEST_TIMEOUT_SECONDS = 600.0
API_MAX_RETRIES = 1

# Process baselines sequentially; local inference has no cloud rate limit.
DATASETS_PER_BATCH = 3
INTER_DATASET_DELAY_SECONDS = 0
INTER_BATCH_DELAY_SECONDS = 0

print("Project root:", PROJECT_ROOT)
print("Top-down inputs:", TOP_DOWN_DIR)
print("Top-down + bottom-up outputs:", TOP_DOWN_BOTTOM_UP_DIR)
print("Review: cosine similarity + one global documentary prompt per model")
print("Qwen generator/reconstructor (LM Studio locale):", llm_clients.QWEN_GENERATOR_MODEL)
print("Bottom-up critic ensemble (LM Studio):", llm_clients.configured_bottom_up_critic_models())
print("Bottom-up ensemble prompting: FEW_SHOT")
print("Generation repair critic (LM Studio):", llm_clients.configured_critic_model())
print("LM Studio endpoint:", llm_clients.LM_STUDIO_BASE_URL)

Project root: C:\Users\agnes\Desktop\tesi originale\LLM_MultiAgentChain_GoalOrientedAPIAlignement
Top-down inputs: C:\Users\agnes\Desktop\tesi originale\LLM_MultiAgentChain_GoalOrientedAPIAlignement\output\top_down
Top-down + bottom-up outputs: C:\Users\agnes\Desktop\tesi originale\LLM_MultiAgentChain_GoalOrientedAPIAlignement\output\top_down_bottom_up
Completeness: sempre attiva; cartelle di output unificate
Qwen generator/reconstructor (LM Studio locale): qwen/qwen3-vl-8b
Bottom-up critic ensemble (LM Studio): ('mistralai/ministral-3-14b-reasoning', 'ibm/granite-3.2-8b', 'google/gemma-4-12b-qat')
Bottom-up ensemble prompting: FEW_SHOT
Completeness critic ensemble (LM Studio): ('mistralai/ministral-3-14b-reasoning', 'ibm/granite-3.2-8b', 'google/gemma-4-12b-qat')
Generation repair critic (LM Studio): mistralai/ministral-3-14b-reasoning
LM Studio endpoint: http://127.0.0.1:1234/v1


## 1. Individuazione degli output top-down

Il notebook usa tutti i file `*_top_down.json` presenti nella cartella della baseline, salvo override espliciti.

In [2]:
if BASELINE_FILE_OVERRIDES:
    baseline_files = [Path(p) for p in BASELINE_FILE_OVERRIDES]
else:
    baseline_files = []
    for path in sorted(TOP_DOWN_DIR.glob("*_top_down.json")):
        with path.open("r", encoding="utf-8") as file:
            dataset_name = json.load(file).get("name")
        if dataset_name in GROUNDTRUTH_NAMES:
            baseline_files.append(path)

missing = [p for p in baseline_files if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing baseline files: {missing}")
if not baseline_files:
    raise FileNotFoundError(
        f"No *_top_down.json found in {TOP_DOWN_DIR}. "
        "Run 01_pipeline_execution_top_down_only.ipynb first."
    )

print(f"Found {len(baseline_files)} top-down baseline file(s):")
for p in baseline_files:
    print("-", p.name)

Found 12 top-down baseline file(s):
- SIA Project 25 26_ZERO_SHOT_feedback_original_top_down_top_down.json
- Assegno Unico Universale - SIA Project 24 25_ZERO_SHOT_feedback_original_top_down_top_down.json
- La Reine Marlene - SIA Project 23 24_ZERO_SHOT_feedback_original_top_down_top_down.json
- Ethical Purchasing Group - SIA Project 22 23_ZERO_SHOT_feedback_original_top_down_top_down.json
- Event Organization Portal - SIA Project 21 22_ZERO_SHOT_feedback_original_top_down_top_down.json
- Calculator_ZERO_SHOT_feedback_original_top_down_top_down.json
- Fitness Tracker_ZERO_SHOT_feedback_original_top_down_top_down.json
- Genome Nexus_ZERO_SHOT_feedback_original_top_down_top_down.json
- Gestao Hospital_ZERO_SHOT_feedback_original_top_down_top_down.json
- London Ambulance Service_ZERO_SHOT_feedback_original_top_down_top_down.json
- Planning Poker_ZERO_SHOT_feedback_original_top_down_top_down.json
- Recycling_ZERO_SHOT_feedback_original_top_down_top_down.json


## 2. Caricamento dello stato iniziale

I file top-down contengono direttamente `structuredHighLevelGoals` e `structuredLowLevelGoals`. Il ciclo riusa quindi le relazioni padre-figlio realmente prodotte dalla pipeline top-down, senza mapping hard-coded. Le eventuali riscritture HLG e le rigenerazioni LLG usano una configurazione bottom-up esplicita: generazione Zero-shot con critic Few-shot attivo. La configurazione della baseline viene conservata come provenienza, ma non controlla le nuove generazioni del ciclo.

In [3]:
from src.data_model import Actors, HighLevelGoals, LowLevelGoals
from src.examples.shot_learning import ShotPromptingMode

BOTTOM_UP_GENERATION_PROMPTING_MODE = ShotPromptingMode.ZERO_SHOT
BOTTOM_UP_GENERATION_EVALUATOR_ABLATION = False
BOTTOM_UP_GENERATION_CRITIC_FEW_SHOT_EXAMPLES = 4


def _infer_mode_from_name(path: Path) -> str:
    match = re.search(r"(?:^|_)(ZERO_SHOT|ONE_SHOT|FEW_SHOT)(?=_|$)", path.stem.upper())
    if not match:
        raise ValueError(f"Cannot infer prompting mode from {path.name}")
    return match.group(1)


def _stage_generation_config(
    config: dict,
    stage_name: str,
    baseline_file: Path,
) -> tuple[ShotPromptingMode, bool]:
    stage = config.get(stage_name) or {}
    mode_name = stage.get("promptingMode") or _infer_mode_from_name(
        baseline_file
    )
    try:
        mode = ShotPromptingMode[str(mode_name).upper()]
    except KeyError as exc:
        raise ValueError(
            f"Unsupported {stage_name} prompting mode {mode_name!r} in "
            f"{baseline_file.name}"
        ) from exc
    evaluator_ablation = not bool(stage.get("evaluator", True))
    return mode, evaluator_ablation


def load_top_down_state(baseline_file: Path):
    with baseline_file.open("r", encoding="utf-8") as f:
        payload = json.load(f)

    description = payload["description"]
    config = payload.get("pipelineConfiguration") or {}
    hlg_mode, hlg_evaluator_ablation = _stage_generation_config(
        config, "highLevelGoals", baseline_file
    )
    llg_mode, llg_evaluator_ablation = _stage_generation_config(
        config, "lowLevelGoals", baseline_file
    )
    structured_hlg = payload.get("structuredHighLevelGoals")
    structured_llg = payload.get("structuredLowLevelGoals")

    if structured_hlg is None or structured_llg is None:
        raise ValueError(
            f"{baseline_file.name} does not contain structured top-down goals. "
            "Run 01_pipeline_execution_top_down_only.ipynb first."
        )

    high_level_goals = HighLevelGoals.model_validate(structured_hlg)
    low_level_goals = LowLevelGoals.model_validate(structured_llg)

    return {
        "payload": payload,
        "description": description,
        "high_level_goals": high_level_goals,
        "actors": Actors.model_validate(payload["structuredActors"]) if payload.get("structuredActors") else None,
        "low_level_goals": low_level_goals,
        "source_hlg_prompting_mode": hlg_mode,
        "source_llg_prompting_mode": llg_mode,
        "source_hlg_llama_ablation": hlg_evaluator_ablation,
        "source_llg_llama_ablation": llg_evaluator_ablation,
        "prompting_mode": BOTTOM_UP_GENERATION_PROMPTING_MODE,
        "llama_ablation": BOTTOM_UP_GENERATION_EVALUATOR_ABLATION,
        "source_kind": "structured baseline fields",
        "mapped_input_file": None,
    }

## 3. Cosine similarity e revisione globale

Ogni iterazione salva soltanto `<progetto>_iteration_NNN.json` e `<progetto>_cosine_NNN.json`. Il secondo file contiene tutti gli HLG e HLG′, i confronti col proprio padre e con gli HLG originali dello stesso attore, applicando il filtro prima del calcolo. Lo stesso oggetto JSON viene passato identico ai tre evaluator insieme alla documentazione. Gli evaluator possono scegliere soltanto KEEP, REWRITE, REMOVE o DUPLICATE_REMOVAL; non vengono aggiunti nuovi HLG. Se due HLG′ dello stesso attore raggiungono 0.80, entrambi ricevono `duplicate_label: "DUPLICATE"` e gli identificativi in `duplicate_candidates`; il confronto con il proprio padre resta separato. La rimozione richiede comunque verifica documentale e quorum.

Il notebook stampa i log dettagliati: confronti cosine, modello e tentativo in corso, decisione e motivazione per ogni branch, errori di validazione, quorum e interventi applicati. Risposte e tentativi sono registrati anche in `<progetto>_iteration_NNN.json`, senza file separati per modello o branch. La soglia del padre è 0.68, preliminare su otto coppie; quella dei duplicati resta 0.80. Rieseguire la cella di import/reload prima del ciclo.

Gli evaluator restituiscono JSON vincolato da schema. I retry richiedono solo i branch irrisolti, mantenendo il contesto globale; la cronologia registra motivo di arresto e uso dei token. Rieseguire import/reload per attivare questa configurazione.


In [ ]:
import shutil
import importlib
import src.data_model as _data_model
import src.extraction.extractor as _extractor
import src.bottom_up.goal_reconstructor as _goal_reconstructor
import src.bottom_up.goal_similarity as _goal_similarity
import src.bottom_up.global_goal_evaluator as _global_goal_evaluator
import src.bottom_up.goal_cycle_orchestrator as _bottom_up_orchestrator
import src.self_critique.refine_response as _reflection
importlib.reload(_data_model)
importlib.reload(llm_clients)
importlib.reload(_extractor)
importlib.reload(_goal_reconstructor)
importlib.reload(_reflection)
importlib.reload(_goal_similarity)
importlib.reload(_global_goal_evaluator)
importlib.reload(_bottom_up_orchestrator)
from src.data_model import Actors, HighLevelGoals, LowLevelGoals
from src.bottom_up.goal_cycle_orchestrator import (
    run_global_goal_cycle,
    DEFAULT_GLOBAL_CYCLE_MAX_ITERATIONS,
    BOTTOM_UP_CRITIC_MEMORY_TURNS,
    BOTTOM_UP_CRITIC_STATE_UPDATES,
)
from src.self_critique.refine_response import (
    BOTTOM_UP_MAX_ATTEMPTS as MAX_ATTEMPTS,
    BOTTOM_UP_QUALITY_THRESHOLD as QUALITY_THRESHOLD,
)
_reflection.BOTTOM_UP_EVALUATOR_FEW_SHOT_EXAMPLE_COUNT = (
    BOTTOM_UP_GENERATION_CRITIC_FEW_SHOT_EXAMPLES
)


def _close_client_safely(client):
    if client is None:
        return
    try:
        client.close()
    except Exception as exc:
        print(f"Warning while closing client: {type(exc).__name__}: {exc}")


def reset_model_clients():
    llm_clients.REQUEST_TIMEOUT_SECONDS = REQUEST_TIMEOUT_SECONDS
    llm_clients.API_MAX_RETRIES = API_MAX_RETRIES
    llm_clients.reset_model_clients()


def close_model_clients():
    _close_client_safely(getattr(llm_clients, "qwen_client", None))
    _close_client_safely(getattr(llm_clients, "ministral_client", None))
    llm_clients.qwen_client = None
    llm_clients.ministral_client = None


def bottom_up_model_configuration() -> dict:
    return {
        "generatorProvider": "LM Studio (local)",
        "generatorModel": llm_clients.QWEN_GENERATOR_MODEL,
        "evaluatorProvider": "LM Studio (local)",
        "consoleVerbosity": "detailed",
        "evaluatorResponseFormat": "json_schema",
        "evaluatorEndpoint": "/v1/chat/completions",
        "iterationOutputFiles": ["<progetto>_iteration_NNN.json", "<progetto>_cosine_NNN.json"],
        "evaluatorModels": list(_global_goal_evaluator.BOTTOM_UP_EVALUATOR_MODELS),
        "evaluatorPromptingMode": _global_goal_evaluator.BOTTOM_UP_EVALUATOR_PROMPTING_MODE,
        "critic1FewShotExamples": _global_goal_evaluator.CRITIC1_FEW_SHOT_EXAMPLE_COUNT,
        "generationReflectionEvaluatorModel": llm_clients.configured_critic_model(),
        "evaluatorVotingQuorum": _global_goal_evaluator.EVALUATOR_VOTING_QUORUM,
        "critic1Policy": "ONE_GLOBAL_DOCUMENTARY_RESPONSE_WITH_FIXED_COSINE_MATCHES",
        "reconstructionRelationships": ["EQUIVALENT", "NOT_EQUIVALENT"],
        "similarity": _goal_similarity.similarity_configuration(),
        "completenessEnabled": False,
        "structuralActionPriority": "REMOVALS_THEN_REWRITES_THEN_DIRECT_LLG_REGENERATION",
        "parentReferencePolicy": "EXACT_PARENT_REFERENCE_NO_REPAIR",
        "branchConfirmationPolicy": "REEVALUATE_ALL_BRANCHES_EVERY_ITERATION",
        "evaluatorContextLength": llm_clients.LM_STUDIO_EVALUATOR_CONTEXT_LENGTH,
        "evaluatorVotingTemperature": _global_goal_evaluator.BOTTOM_UP_EVALUATOR_TEMPERATURE,
        "evaluatorVoteAttempts": _global_goal_evaluator.BOTTOM_UP_EVALUATOR_VOTE_ATTEMPTS,
        "datasetBatchSize": DATASETS_PER_BATCH,
        "interDatasetDelaySeconds": INTER_DATASET_DELAY_SECONDS,
        "interBatchDelaySeconds": INTER_BATCH_DELAY_SECONDS,
        "hlgGenerationPromptingMode": BOTTOM_UP_GENERATION_PROMPTING_MODE.name,
        "llgGenerationPromptingMode": BOTTOM_UP_GENERATION_PROMPTING_MODE.name,
        "hlgGenerationEvaluatorEnabled": not BOTTOM_UP_GENERATION_EVALUATOR_ABLATION,
        "llgGenerationEvaluatorEnabled": not BOTTOM_UP_GENERATION_EVALUATOR_ABLATION,
        "generationCriticFewShotExamples": BOTTOM_UP_GENERATION_CRITIC_FEW_SHOT_EXAMPLES,
        "maxIterations": (
            DEFAULT_GLOBAL_CYCLE_MAX_ITERATIONS
            if GLOBAL_CYCLE_MAX_ITERATIONS_OVERRIDE is None
            else GLOBAL_CYCLE_MAX_ITERATIONS_OVERRIDE
        ),
        "qualityThreshold": QUALITY_THRESHOLD,
        "maxReflectionAttempts": MAX_ATTEMPTS,
        "pipelinePolicyVersion": _global_goal_evaluator.PIPELINE_POLICY_VERSION,
        "criticMemoryTurns": BOTTOM_UP_CRITIC_MEMORY_TURNS,
        "criticStateUpdates": BOTTOM_UP_CRITIC_STATE_UPDATES,
        "evaluatorMaxCompletionTokens": llm_clients.BOTTOM_UP_CRITIC_MAX_TOKENS,
        "maxHlgRewritesPerLineage": _bottom_up_orchestrator.MAX_HLG_REWRITES_PER_BRANCH,
    }


def combined_output_file_for(baseline_file: Path) -> Path:
    base = baseline_file.stem.removesuffix("_top_down")
    return TOP_DOWN_BOTTOM_UP_DIR / f"{base}_top_down_bottom_up.json"


def reset_iteration_outputs(project_root: Path, iterations_root: Path) -> Path:
    """Clear only the project's iteration directory before a new batch starts."""
    project_root = Path(project_root).resolve()
    expected = project_root / "output" / "bottom_up_iterations"
    target = Path(iterations_root).absolute()

    # Validate the absolute deletion target before invoking recursive removal.
    # Resolved equality also rejects redirected output parents and junctions.
    if target != expected or target.resolve() != expected or target.is_symlink():
        raise ValueError("Cleanup is restricted to PROJECT_ROOT/output/bottom_up_iterations")
    if target.exists():
        if not target.is_dir():
            raise ValueError("Iteration output path must be a directory")
        shutil.rmtree(target)
    target.mkdir(parents=True, exist_ok=True)
    return target


def project_iteration_directory(iterations_root: Path, project_name: str) -> Path:
    """Preserve readable project names while preventing path traversal."""
    name = re.sub(r'[<>:"/\\|?*\x00-\x1f]', "_", project_name).strip().rstrip(". ")
    if not name or name in {".", ".."}:
        raise ValueError("A non-empty project name is required")

    root = Path(iterations_root).resolve()
    directory = root / name
    if directory.resolve().parent != root or directory.is_symlink():
        raise ValueError("Project directory must stay inside the iteration output root")
    return directory


def iteration_directory_for(project_name: str) -> Path:
    return project_iteration_directory(ITERATIONS_ROOT, project_name)


## 4. Esecuzione bottom-up

Ogni dataset viene processato in modo indipendente. L'output combinato parte da una copia del JSON top-down e aggiunge il trace completo del ciclo e le collezioni finali. Il file top-down di partenza non viene mai sovrascritto.



In [ ]:

cycle_results = {}
cycle_errors = {}

max_iterations = (
    DEFAULT_GLOBAL_CYCLE_MAX_ITERATIONS
    if GLOBAL_CYCLE_MAX_ITERATIONS_OVERRIDE is None
    else int(GLOBAL_CYCLE_MAX_ITERATIONS_OVERRIDE)
)

def wait_before_next_dataset(dataset_index: int, total_datasets: int) -> None:
    if dataset_index >= total_datasets:
        return
    batch_complete = dataset_index % DATASETS_PER_BATCH == 0
    delay_seconds = INTER_BATCH_DELAY_SECONDS if batch_complete else INTER_DATASET_DELAY_SECONDS
    reason = "batch pause" if batch_complete else "dataset pause"
    print(f"\n[{reason}] waiting {delay_seconds} seconds before the next dataset...")
    time.sleep(delay_seconds)


def run_profile(baseline_file: Path, state: dict):
    baseline_payload = state["payload"]
    dataset_name = baseline_payload["name"]
    profile_name = _global_goal_evaluator.PIPELINE_POLICY_VERSION
    result_key = f"{dataset_name} [{profile_name}]"
    iteration_dir = iteration_directory_for(dataset_name)
    iteration_dir.mkdir(parents=True, exist_ok=True)
    print(f"\n[{dataset_name}] starting {profile_name}")
    print("  input:", baseline_file.name)
    print("  similarity thresholds: parent=", _goal_similarity.HLG_SIMILARITY_THRESHOLD, "duplicates=", _goal_similarity.DUPLICATE_SIMILARITY_THRESHOLD)
    print("  initial HLG:", len(state["high_level_goals"].goals))
    print("  initial LLG:", len(state["low_level_goals"].low_level_goals))

    started_at = time.perf_counter()
    result = None
    error = None
    try:
        reset_model_clients()
        result = run_global_goal_cycle(
            project_description=state["description"],
            actors=state["actors"],
            initial_high_level_goals=state["high_level_goals"],
            initial_low_level_goals=state["low_level_goals"],
            mode=state["prompting_mode"],
            evaluator_ablation=state["llama_ablation"],
            evaluation_output_directory=iteration_dir,
            output_file_prefix=dataset_name,
            max_iterations=max_iterations,
            verbose=True,
        )
    except Exception as exc:
        error = f"{type(exc).__name__}: {exc}"
    finally:
        close_model_clients()
        print(f"  client resources closed after {time.perf_counter() - started_at:.1f}s")

    output_file = combined_output_file_for(baseline_file)
    payload = {key: value for key, value in baseline_payload.items() if key != "hierarchyAudit"}
    payload.update({
        "baselineSourceFile": baseline_file.name,
        "bottomUpRunProfile": profile_name,
        "bottomUpInputSource": state["source_kind"],
        "bottomUpModelConfiguration": {**bottom_up_model_configuration(),
            "resolvedSimilarityConfiguration": result.configuration.get("similarity") if result else None},
        "evaluationOutputDirectory": str(iteration_dir),
    })
    if error is not None:
        cycle_errors[result_key] = error
        payload["bottomUpError"] = error
    else:
        cycle_results[result_key] = result
        payload.update({
            "globalGoalCycle": result.model_dump(mode="json"),
            "finalHighLevelGoals": result.final_high_level_goals.model_dump(mode="json"),
            "finalLowLevelGoals": result.final_low_level_goals.model_dump(mode="json"),
        })

    with output_file.open("w", encoding="utf-8") as f:
        json.dump(payload, f, indent=4, ensure_ascii=False)
    if error is not None:
        print(f"  FAILED: {error} | saved: {output_file.name}")
    else:
        print(
            f"  saved: {output_file.name} | converged={result.converged}, "
            f"stop_reason={result.stop_reason.value}, iterations={result.completed_iterations}, "
            f"total HLG={len(result.final_high_level_goals.goals)}, "
            f"total LLG={len(result.final_low_level_goals.low_level_goals)}"
        )


def run_all_datasets() -> None:
    reset_iteration_outputs(PROJECT_ROOT, ITERATIONS_ROOT)
    cycle_results.clear()
    cycle_errors.clear()
    print("Iteration outputs cleared for this execution:", ITERATIONS_ROOT)
    profile_name = "COSINE + DOCUMENTARY REVIEW"
    total_batches = (len(baseline_files) + DATASETS_PER_BATCH - 1) // DATASETS_PER_BATCH
    print(f"\n===== BOTTOM-UP {profile_name} =====")
    for dataset_index, baseline_file in enumerate(baseline_files, start=1):
        batch_number = (dataset_index - 1) // DATASETS_PER_BATCH + 1
        print(f"\n=== DATASET {dataset_index}/{len(baseline_files)} | BATCH {batch_number}/{total_batches} ===")
        try:
            state = load_top_down_state(baseline_file)
        except Exception as exc:
            dataset_name = baseline_file.stem
            error = f"INPUT_LOAD_ERROR - {type(exc).__name__}: {exc}"
            profile_key = _global_goal_evaluator.PIPELINE_POLICY_VERSION
            cycle_errors[f"{dataset_name} [{profile_key}:input]"] = error
            print(f"[{dataset_name}] FAILED while loading top-down state: {exc}")
            wait_before_next_dataset(dataset_index, len(baseline_files))
            continue

        run_profile(baseline_file, state)
        wait_before_next_dataset(dataset_index, len(baseline_files))

### 4. Esecuzione della revisione

All'avvio di `run_all_datasets()` vengono cancellati i trace delle esecuzioni precedenti e le relative cartelle sotto `output/bottom_up_iterations/`. Ogni progetto usa poi una cartella con il proprio nome, per esempio `SIA Project 25 26`, contenente tutte le iterazioni e le cosine dell'esecuzione corrente. La pulizia non avviene tra iterazioni. Le baseline top-down restano invariate. Le celle mostrate possono contenere output storici: la configurazione effettiva viene registrata per ogni nuova esecuzione.


In [ ]:
run_all_datasets()



===== BOTTOM-UP ANALISI DI COMPLETEZZA =====

=== DATASET 1/12 | BATCH 1/4 ===

[SIA Project 25 26] starting with_completeness
  input: SIA Project 25 26_ZERO_SHOT_feedback_original_top_down_top_down.json
  analisi di completezza: obbligatoria
  initial HLG: 8
  initial LLG: 13
[bottom-up] iteration 1/25 starting
[bottom-up] full pass: 8 active branch(es) out of 8 total
[bottom-up] loading complete critic pass: mistralai/ministral-3-14b-reasoning
[bottom-up] mistralai/ministral-3-14b-reasoning: Critic 1 global HLG list (8 branches)
[bottom-up] unloaded complete critic pass: mistralai/ministral-3-14b-reasoning
[bottom-up] loading complete critic pass: ibm/granite-3.2-8b
[bottom-up] ibm/granite-3.2-8b: Critic 1 global HLG list (8 branches)
[bottom-up] unloaded complete critic pass: ibm/granite-3.2-8b
[bottom-up] loading complete critic pass: google/gemma-4-12b-qat
[bottom-up] google/gemma-4-12b-qat: Critic 1 global HLG list (8 branches)
[bottom-up] unloaded complete critic pass: google/

## 5. Confronto strutturale rapido

Questa sezione mostra solo le cardinalità prima/dopo e lo stato di convergenza. **Precision, Recall e F1 non vengono calcolati qui**: sono volutamente delegati al terzo notebook, che confronta il risultato bottom-up con la rispettiva baseline top-down.

In [ ]:
print("\nBottom-up execution summary")
print("---------------------------")
print(f"Inputs considered: {len(baseline_files)}")
print(f"Cycles completed: {len(cycle_results)}")
print(f"Cycles failed: {len(cycle_errors)}")
print(f"Bottom-up outputs (cosine + documentary review): {TOP_DOWN_BOTTOM_UP_DIR}")
print(f"Iteration traces: {ITERATIONS_ROOT}")

for dataset_name, result in cycle_results.items():
    print(
        f"- {dataset_name}: converged={result.converged}, "
        f"stop_reason={result.stop_reason}, "
        f"iterations={result.completed_iterations}, "
        f"final HLG={len(result.final_high_level_goals.goals)}, "
        f"final LLG={len(result.final_low_level_goals.low_level_goals)}"
    )
    if result.warnings:
        print("  warnings:", len(result.warnings), "(details in iteration JSON)")

if cycle_errors:
    print("\nErrors:")
    for dataset_name, error in cycle_errors.items():
        print(f"- {dataset_name}: {error}")


Bottom-up execution summary
---------------------------
Inputs considered: 12
Cycles completed: 12
Cycles failed: 0
Bottom-up outputs (analisi di completezza): C:\Users\agnes\Desktop\tesi originale\LLM_MultiAgentChain_GoalOrientedAPIAlignement\output\top_down_bottom_up
Iteration traces: C:\Users\agnes\Desktop\tesi originale\LLM_MultiAgentChain_GoalOrientedAPIAlignement\output\bottom_up_iterations
- SIA Project 25 26 [with_completeness]: converged=True, stop_reason=GlobalGoalCycleStopReason.ALL_BRANCHES_CONFIRMED, iterations=12, final HLG=10, final LLG=37
  warnings: LLG_REGENERATION_BATCH_LIMIT (4 eligible, 2 scheduled); hlg_bf787da4fb4e0b50_001: REPAIR_REJECTED_RETRY_NEXT_ITERATION: Top-down repair critic did not approve the candidate after 3 generation attempts (score=2.0): The current low-level goals (LLGs) are heavily misaligned with the parent high-level goal (HLG). The HLG focuses solely on making Participium open-source and available to Italian public administrations, while mos